# Destilar la voz **Alex** (Kokoro) a Piper
Timbre de Alex, sin grabar nada. Guarda en tu Google Drive y **retoma si Colab se corta**.
Entrenamiento: fork `KiON-GiON/piper1-gpl@fixes` (comandos verificados contra el código real).

**Antes:** Entorno de ejecución → Cambiar tipo → **T4 GPU**.
**Celda 1** prepara · **Celda 2** entrena (reejecutable) · **Celda 3** exporta y descarga.

In [ ]:
#@title 1. PREPARAR — reusa el dataset del Drive, muestra progreso y avisa con un tono
import IPython, torch, os, time, shutil
from IPython.display import Javascript, Audio, display

t0 = time.time()
def paso(msg):
    print(f"\n[{(time.time()-t0)/60:5.1f} min] === {msg} ===", flush=True)

# Anti-desconexion (clic automatico cada 60s)
display(Javascript('function _keep(){ var b=document.querySelector("colab-toolbar-button#connect"); if(b) b.click() } setInterval(_keep, 60000)'))

assert torch.cuda.is_available(), "Activa T4: Entorno de ejecucion -> Cambiar tipo de entorno -> GPU"
print("GPU:", torch.cuda.get_device_name(0))

from google.colab import drive
drive.mount('/content/drive', force_remount=True)
WORK = "/content/drive/MyDrive/LoudVox/alex"
os.makedirs(WORK, exist_ok=True)
print("Carpeta de trabajo en Drive:", WORK)

DATASET = "/content/dataset"
DATA_ZIP = WORK + "/dataset_alex.zip"

# --- 1) DATASET: si ya esta en el Drive, se recupera; si no, se genera y se guarda ---
if os.path.exists(DATA_ZIP):
    paso("Recuperando el dataset del Drive (NO regenero audios)")
    shutil.unpack_archive(DATA_ZIP, DATASET)
    print(f"Dataset recuperado del Drive: {len(os.listdir(DATASET + '/wavs'))} audios. (Te ahorras ~1 hora.)")
else:
    paso("Generando el dataset con la voz Alex (SOLO la primera vez, tarda ~1 h)")
    !pip install -q kokoro-onnx==0.5.0
    !wget -q -nc -O /content/kokoro.onnx "https://github.com/thewh1teagle/kokoro-onnx/releases/download/model-files-v1.0/kokoro-v1.0.onnx"
    !wget -q -nc -O /content/voices.bin "https://github.com/thewh1teagle/kokoro-onnx/releases/download/model-files-v1.0/voices-v1.0.bin"
    !wget -q -O /content/corpus.txt "https://raw.githubusercontent.com/Lazy-Money/Loud-Web/claude/readvox-research-slumjx/colab/corpus/corpus_es_1300.txt"
    import wave, numpy as np
    from kokoro_onnx import Kokoro
    frases = [l.strip() for l in open("/content/corpus.txt", encoding="utf-8") if l.strip()]
    print(f"{len(frases)} frases a generar. Vas a ver el avance cada 50.")
    kokoro = Kokoro("/content/kokoro.onnx", "/content/voices.bin")
    os.makedirs(DATASET + "/wavs", exist_ok=True)
    rows, total, tg = [], 0.0, time.time()
    for i, f in enumerate(frases):
        if i and i % 50 == 0:
            el = time.time() - tg
            eta = el / i * (len(frases) - i) / 60
            print(f"  {i}/{len(frases)} frases  ·  {el/60:4.1f} min hechos  ·  faltan ~{eta:2.0f} min", flush=True)
        try:
            s, r = kokoro.create(f, voice="em_alex", speed=1.0, lang="es")
        except Exception:
            continue
        idx = np.linspace(0, len(s)-1, int(len(s)*22050/r))
        d = np.clip(np.interp(idx, np.arange(len(s)), s)*32767, -32768, 32767).astype(np.int16)
        if not 1.0 <= len(d)/22050 <= 20.0:
            continue
        n = f"f{i:05d}.wav"
        with wave.open(f"{DATASET}/wavs/{n}", "wb") as w:
            w.setnchannels(1); w.setsampwidth(2); w.setframerate(22050); w.writeframes(d.tobytes())
        rows.append(f"{n}|{f}"); total += len(d)/22050
    open(DATASET + "/metadata.csv", "w", encoding="utf-8").write("\n".join(rows) + "\n")
    print(f"Dataset generado: {len(rows)} clips, {total/60:.1f} min de audio.")
    paso("Guardando el dataset en tu Drive (para no regenerarlo NUNCA mas)")
    shutil.make_archive(WORK + "/dataset_alex", "zip", DATASET)
    print(f"Guardado en Drive: dataset_alex.zip ({os.path.getsize(DATA_ZIP)/1e6:.0f} MB)")

# --- 2) INSTALAR Piper (fork verificado), avisando cada paso ---
paso("Instalando paquetes del sistema (apt)")
!apt-get -q update -y > /dev/null 2>&1
!apt-get -q install -y build-essential cmake ninja-build espeak-ng aria2 > /dev/null 2>&1
paso("Clonando el repo de entrenamiento")
%cd /content
![ -d piper1-gpl ] || git clone -q -b fixes https://github.com/KiON-GiON/piper1-gpl.git
%cd /content/piper1-gpl
paso("Instalando piper con pip (~2-3 min, sin salida mientras trabaja)")
!python -m pip install -q -e .[train]
paso("Compilando el alineador y ajustando protobuf")
!bash build_monotonic_align.sh > /dev/null 2>&1
!pip install -q --upgrade gdown scikit-build protobuf==3.20.3
!python setup.py build_ext --inplace > /dev/null 2>&1

# --- 3) CHECKPOINT base: cacheado en Drive tambien ---
BASE_DRIVE = WORK + "/base.ckpt"
if os.path.exists(BASE_DRIVE) and os.path.getsize(BASE_DRIVE) > 10e6:
    paso("Recuperando el checkpoint base del Drive")
    shutil.copy(BASE_DRIVE, "/content/base.ckpt")
else:
    paso("Descargando el checkpoint base (espanol davefx) desde Hugging Face")
    from huggingface_hub import hf_hub_download
    _b = hf_hub_download(repo_id="rhasspy/piper-checkpoints", repo_type="dataset", filename="es/es_ES/davefx/medium/epoch=5629-step=1605020.ckpt")
    shutil.copy(_b, "/content/base.ckpt")
    shutil.copy("/content/base.ckpt", BASE_DRIVE)
_mb = os.path.getsize("/content/base.ckpt") / 1e6
print(f"base.ckpt: {_mb:.1f} MB")
assert _mb > 10, "La descarga del checkpoint base fallo. Volve a correr la celda 1."
torch.load("/content/base.ckpt", map_location="cpu", weights_only=False)
print("Checkpoint base OK.")

# --- LISTO + tono para avisarte ---
paso(f"LISTO en {(time.time()-t0)/60:.1f} min total. Corre la celda 2 para entrenar.")
import numpy as _np
_sr = 22050; _tt = _np.linspace(0, 0.7, int(_sr*0.7))
_beep = 0.3*_np.sin(2*_np.pi*880*_tt)*_np.exp(-2.5*_tt) + 0.2*_np.sin(2*_np.pi*1320*_tt)*_np.exp(-2.5*_tt)
display(Audio(_beep, rate=_sr, autoplay=True))

In [ ]:
#@title 2. ENTRENAR — retome VALIDADO (copia local) + backup rotativo seguro. Reejecutable.
import os, glob, time, shutil, threading, zipfile, torch

WORK   = "/content/drive/MyDrive/LoudVox/alex"
LOCAL  = "/content/train_local"           # checkpoints locales durante el entrenamiento
BACKUP = WORK + "/backup"                  # 2 slots en Drive (version_1 ultimo, version_0 respaldo)
V1 = BACKUP + "/version_1.ckpt"
V0 = BACKUP + "/version_0.ckpt"
RESUME_LOCAL = "/content/resume.ckpt"
os.makedirs(BACKUP, exist_ok=True)
MIN = 100_000_000

def _valid(p):
    # chequeo barato: el .ckpt es un zip con su directorio central intacto
    try:
        with zipfile.ZipFile(p) as z: z.namelist()
        return True
    except Exception:
        return False
def _epoch(p):
    try: return int(torch.load(p, map_location="cpu", weights_only=False).get("epoch", -1))
    except Exception: return -1

# --- (una sola vez) sembrar version_1 desde tu mejor checkpoint viejo ---------
if not (os.path.exists(V1) and os.path.getsize(V1) > MIN):
    viejos = [p for p in glob.glob(WORK + "/lightning_logs/version_*/checkpoints/last.ckpt")
              if os.path.getsize(p) > MIN and _valid(p)]
    if viejos:
        mejor = max(viejos, key=_epoch)
        print(f"Sembrando version_1 desde tu mejor checkpoint (epoch {_epoch(mejor)})...")
        shutil.copy(mejor, V1)

# --- retome: copiar de Drive a LOCAL y VALIDAR (con reintentos por si el Drive
#     devuelve una lectura a medio sincronizar). Se entrena desde la copia local,
#     asi el error de "lectura inconsistente del Drive" no puede voltear el arranque.
def _prep(slot):
    if not (os.path.exists(slot) and os.path.getsize(slot) > MIN): return None
    for _ in range(3):
        try:
            shutil.copy(slot, RESUME_LOCAL)
            if _valid(RESUME_LOCAL): return _epoch(RESUME_LOCAL)
        except Exception: pass
        time.sleep(3)
    return None

init = '--model.init_from_checkpoint /content/base.ckpt'
cands = [(V1, "version_1"), (V0, "version_0")]
cands += [(p, "lightning_logs") for p in sorted(
    glob.glob(WORK + "/lightning_logs/version_*/checkpoints/last.ckpt"),
    key=os.path.getmtime, reverse=True)]
for slot, nombre in cands:
    ep = _prep(slot)
    if ep is not None and ep >= 0:
        init = f'--ckpt_path "{RESUME_LOCAL}"'
        print(f"RETOMANDO desde {nombre} (epoch {ep}) [copia local validada]")
        break
    elif os.path.exists(slot):
        print(f"  {nombre}: lectura no valida, pruebo el siguiente...")
else:
    print("Ningun checkpoint valido -> parto del base espanol")

# --- backup rotativo SEGURO: solo sube checkpoints validos; nunca pisa el
#     respaldo con algo roto; verifica lo subido ---------------------------------
_stop = threading.Event()
def _newest_local():
    c = glob.glob(LOCAL + "/lightning_logs/**/checkpoints/last.ckpt", recursive=True)
    return max(c, key=os.path.getmtime) if c else None
def _rotar(local):
    if not (local and os.path.exists(local) and _valid(local)):
        return                                   # nunca subir basura
    if os.path.exists(V1) and _valid(V1):
        shutil.copy(V1, V0)                      # respaldo: version_1 SANO -> version_0
    shutil.copy(local, V1)                       # nuevo -> version_1 (sobrescribe, sin papelera)
    if not _valid(V1):
        shutil.copy(local, V1)                   # si la subida quedo a medias, reintento una vez
    print(f"  [backup {time.strftime('%H:%M:%S')}] Drive: version_1 actualizado (anterior -> version_0)", flush=True)
def _watch():
    visto = 0
    while not _stop.is_set():
        try:
            loc = _newest_local()
            if loc and os.path.getmtime(loc) != visto:
                visto = os.path.getmtime(loc); _rotar(loc)
        except Exception as e:
            print("  [backup] reintenta:", e, flush=True)
        _stop.wait(20)
threading.Thread(target=_watch, daemon=True).start()

# --- entrenar (checkpoints LOCALES cada 5 epochs; el watcher los sube a Drive) --
cmd = (
 'cd /content/piper1-gpl && python -m piper.train fit --data.voice_name "alex" '
 '--data.csv_path /content/dataset/metadata.csv --data.audio_dir /content/dataset/wavs '
 '--data.espeak_voice es --data.cache_dir /content/cache '
 f'--data.config_path "{WORK}/alex.onnx.json" --data.batch_size 12 --model.sample_rate 22050 '
 '--data.validation_split 0 --data.num_test_examples 0 '
 f'--trainer.default_root_dir "{LOCAL}" '
 '--trainer.accelerator gpu --trainer.devices 1 --trainer.max_epochs 10000 '
 '--trainer.precision 16-mixed --checkpoint.save_top_k 0 --checkpoint.monitor null '
 '--last_checkpoint.every_n_epochs 5 ' + init
)
try:
    get_ipython().system(cmd)
finally:
    _stop.set()
    loc = _newest_local()
    if loc:
        print("Backup final..."); _rotar(loc)

In [ ]:
#@title 3. EXPORTAR Y DESCARGAR — sale con el nombre listo para LoudVox (sin renombrar)
import os, json
WORK = "/content/drive/MyDrive/LoudVox/alex"
V1 = WORK + "/backup/version_1.ckpt"; V0 = WORK + "/backup/version_0.ckpt"
ckpt = V1 if (os.path.exists(V1) and os.path.getsize(V1) > 100_000_000) else V0
assert os.path.exists(ckpt), "Todavia no hay backup. Deja correr la celda 2 unos epochs."

VOICE = "es_ES-alex-medium"          # nombre que LoudVox reconoce como voz espanola "Espana - Alex"
onnx  = f"{WORK}/{VOICE}.onnx"
print("Exportando desde:", ckpt)
get_ipython().system(f'cd /content/piper1-gpl && python -m piper.train.export_onnx --checkpoint "{ckpt}" --output-file "{onnx}"')

# Config final con el MISMO nombre; asegura idioma y nombre visible
dst_cfg = f"{onnx}.json"
src_cfg = dst_cfg if os.path.exists(dst_cfg) else (WORK + "/alex.onnx.json")
cfg = json.load(open(src_cfg, encoding="utf-8")) if os.path.exists(src_cfg) else {}
cfg.setdefault("phoneme_map", {})
cfg.setdefault("language", {})
cfg["language"].setdefault("code", "es_ES")   # -> se lista bajo Espanol
cfg["dataset"] = "alex"                        # -> nombre visible "Espana - Alex"
json.dump(cfg, open(dst_cfg, "w", encoding="utf-8"), ensure_ascii=False, indent=2)

from google.colab import files
files.download(onnx); files.download(dst_cfg)
print(f"\nListo. Copia {VOICE}.onnx y {VOICE}.onnx.json a tu carpeta de voces:")
print(r"  Windows: %LOCALAPPDATA%\loudvox\voices")
print("Reinicia LoudVox y elegi 'Espana - Alex' en Configuracion. Sin renombrar nada.")